# DATA266 Lab 1 — Task 1: GPT-style LLM from scratch (Dipin)

Character-level, decoder-only Transformer trained on TinyStories. **No prebuilt Transformer or attention modules**:
multi-head self-attention, the causal mask, LayerNorm, the feed-forward network and residual connections are written by hand in `src/model.py`.

All settings come from `configs/task1_config.json`. Set `DIPIN_T1_SMOKE=1` to run the notebook on a tiny subset (pipeline check only).

In [ ]:
import json, os, sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import Image, Markdown, display

TASK_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "task1_config.json").exists())
sys.path.insert(0, str(TASK_DIR / "src"))

from data import DATA_DIR, REPO_DIR, get_batch, load_config, prepare, set_seed, window_starts
from model import build_model, count_parameters
from train import CKPT_DIR, load_checkpoint, lr_at, train
from evaluate import evaluate, generate_text
from utils import get_device, hardware_info

SMOKE = os.environ.get("DIPIN_T1_SMOKE") == "1"
cfg = load_config()
set_seed(cfg["seed"])
device = get_device()
OUT = TASK_DIR / "outputs" / ("smoke" if SMOKE else "")
print("smoke:", SMOKE)
print(json.dumps(hardware_info(device), indent=2))
print(json.dumps({k: cfg[k] for k in ["seed", "model", "training", "generation"]}, indent=2))

## 1.1 Data preprocessing
- **Split:** 100,000 training and 10,000 validation stories, drawn without overlap from the TinyStories train split with my own seed.
- **Tokenizer:** character level. `char_to_idx` / `idx_to_char` are built from the training stories only. Characters seen fewer than 10 times map to `<unk>`; `<eos>` marks the end of each story.
- **Sequences:** each split is one stream (`story <eos> story <eos> …`). An example is a window of 257 ids: input = first 256, target = the same window shifted by one character.

In [ ]:
data = prepare(cfg, max_stories=300, out_dir=OUT / "data") if SMOKE else prepare(cfg)
tok, meta = data["tokenizer"], data["meta"]

In [ ]:
print("vocab size:", tok.vocab_size)
print("char_to_idx (first 40):", dict(list(tok.char_to_idx.items())[:40]))
print("idx_to_char (first 40):", dict(list(tok.idx_to_char.items())[:40]))
sample = data["example_story"][:120]
ids = tok.encode(sample)
print("\ntext   :", repr(sample))
print("encoded:", ids[:40], "...")
print("decoded == original:", tok.decode(ids) == sample)

T = cfg["data"]["sequence_length"]
stream = torch.from_numpy(data["train"])
x, y = get_batch(stream, window_starts(len(stream), T)[:2], T)
print("\ninput  shape", tuple(x.shape), "| target shape", tuple(y.shape))
print("input [0][:60] :", repr(tok.decode(x[0][:60])))
print("target[0][:60] :", repr(tok.decode(y[0][:60])))
print("target is input shifted by one:", bool((x[:, 1:] == y[:, :-1]).all()))

## 1.2 GPT model
Token embedding + learned positional embedding → 6 pre-LayerNorm Transformer blocks (8-head causal self-attention, GELU feed-forward, residual connections) → final LayerNorm → language-model head tied to the token embedding.

In [ ]:
model = build_model(cfg, tok.vocab_size).to(device)
print(model)
print(f"\ntrainable parameters: {count_parameters(model):,}")
per = {n: sum(p.numel() for p in m.parameters()) for n, m in [("token embedding (tied with lm_head)", model.tok_emb),
       ("positional embedding", model.pos_emb), ("one transformer block", model.blocks[0]), ("final LayerNorm", model.ln_f)]}
display(pd.Series(per, name="parameters").to_frame())

In [ ]:
# Causal-mask check: changing the token at position 100 must not change any logit before position 100.
model.eval()
xb = x.to(device)
x2 = xb.clone(); x2[:, 100] = (x2[:, 100] + 1) % tok.vocab_size
with torch.no_grad():
    a, loss0 = model(xb, y.to(device)); b, _ = model(x2)
print("logits shape:", tuple(a.shape), "= (batch, sequence, vocab)")
print("positions < 100 unchanged:", torch.allclose(a[:, :100], b[:, :100], atol=1e-5))
print("positions >= 100 changed :", not torch.allclose(a[:, 100:], b[:, 100:]))
print(f"untrained loss {loss0.item():.3f} vs ln(vocab) {torch.log(torch.tensor(float(tok.vocab_size))).item():.3f}")
with torch.no_grad():
    _, w = model.blocks[0].attn(model.ln_f(model.tok_emb(xb[:1])), return_weights=True)
print("attention weights above the diagonal (future positions) sum to:", float(torch.triu(w[0, 0], 1).sum()))
del model

## 1.3 Training and text generation
Cross-entropy loss, AdamW, linear LR warm-up followed by cosine decay, gradient clipping, 10 epochs.
Training resumes from `checkpoints/latest.pt` if a previous session was interrupted; the raw JSONL log is append-only.

In [ ]:
import matplotlib.pyplot as plt
total = 1000 if SMOKE else cfg["training"]["epochs"] * (meta["train"]["sequences_per_epoch"] // cfg["training"]["batch_size"])
plt.figure(figsize=(6, 3)); plt.plot([lr_at(s, cfg["training"], total) for s in range(0, total, max(1, total // 500))])
plt.title("Learning-rate schedule (warm-up + cosine)"); plt.xlabel("step (x total/500)"); plt.ylabel("lr"); plt.tight_layout(); plt.show()

In [ ]:
summary = train(cfg, data, device, epochs=2 if SMOKE else None, smoke=SMOKE, resume=True)
display(pd.DataFrame(summary["history"]).set_index("epoch"))

In [ ]:
result, rows = evaluate(smoke=SMOKE, data=data)
display(Image(filename=str(OUT / "loss_curves.png")))
if (OUT / "training_stability.png").exists():
    display(Image(filename=str(OUT / "training_stability.png")))

### Generated text: temperature sampling and greedy decoding

In [ ]:
ckpt = CKPT_DIR / ("smoke/model.pt" if SMOKE else "model.pt")
model, tok, _ = load_checkpoint(ckpt, device)
torch.manual_seed(cfg["seed"])
for temp in [0.5, 0.8, 1.0]:
    print(f"--- temperature {temp} ---")
    print(generate_text(model, tok, "Once upon a time", 400, temp, device, fixed_length=False), "\n")
print("--- greedy ---")
print(generate_text(model, tok, "Once upon a time", 400, 1.0, device, greedy=True, fixed_length=False))

In [ ]:
for s in result["generation"]["samples"][::10 if not SMOKE else 1]:
    print(f"[sample {s['id']}] {s['text']}\n")

## Evaluation metrics (all required metrics)

In [ ]:
pd.set_option("display.max_colwidth", 200)
display(pd.DataFrame(rows, columns=["metric", "value", "notes"]).set_index("metric"))
g = result["generation"]
display(pd.DataFrame({"character-level": g["char_level"], "word-level": g["word_level"]}))

## 1.4 Failure analysis: candidates
Samples flagged automatically by simple heuristics (repetition, words never seen in training). The three cases, their failure types and my observations are written by hand in `failure_analysis.md`.

In [ ]:
display(Markdown((OUT / "failure_candidates.md").read_text(encoding="utf-8")))

## Reproducibility manifest

In [ ]:
import platform, subprocess
env = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
(OUT / "environment.txt").write_text(f"python {platform.python_version()}\ntorch {torch.__version__}\n{env}")
manifest = {"task": "task1_llm", "member": "dipin", "run_id": summary["run_id"],
            "config": "task1_llm/dipin/configs/task1_config.json", "notebook": "task1_llm/dipin/src/task1_dipin.ipynb",
            "raw_log": summary["raw_log"], "checkpoints": summary["checkpoints"], "reported_checkpoint": result["checkpoint"],
            "metrics_file": "task1_llm/dipin/metrics_report.csv", "evaluation": "task1_llm/dipin/outputs/evaluation.json",
            "environment": "reproducibility/manifests/dipin/environment_task1.txt", "hardware": summary["hardware"],
            "validation_ce": result["validation"]["ce_nats"], "training_ce": result["train"]["ce_nats"]}
if not SMOKE:
    man_dir = REPO_DIR / "reproducibility" / "manifests" / "dipin"
    (man_dir / "task1_manifest.json").write_text(json.dumps(manifest, indent=2))
    (man_dir / "environment_task1.txt").write_text((OUT / "environment.txt").read_text())
print(json.dumps(manifest, indent=2))